In [1]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/yasserh/student-marks-dataset/Student_Marks.csv


In [2]:
train=pd.read_csv("/kaggle/input/datasets/yasserh/student-marks-dataset/Student_Marks.csv")

In [3]:
print(train.shape)
print(train.columns)
print(train.isna().sum())
print(train.dtypes)

(100, 3)
Index(['number_courses', 'time_study', 'Marks'], dtype='object')
number_courses    0
time_study        0
Marks             0
dtype: int64
number_courses      int64
time_study        float64
Marks             float64
dtype: object


In [4]:
X=train.drop(['Marks'],axis=1)
y=train['Marks']

In [5]:
y.shape

(100,)

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2)

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.model_selection import cross_val_score

linear_pipeline=Pipeline(
    steps=[
        ("linear",LinearRegression())        
    ]
)
ridge_pipeline=Pipeline(
    steps=[
        ("ridge",Ridge(alpha=0.1))
    ]
)
lasso_pipeline=Pipeline(
    steps=[
        ("lasso",Lasso(alpha=0.1))
    ]
)

linear_pipeline.fit(X_train, y_train)
ridge_pipeline.fit(X_train, y_train)
lasso_pipeline.fit(X_train,y_train)

Pipeline(steps=[('lasso', Lasso(alpha=0.1))])

In [8]:
linear_predict=linear_pipeline.predict(X_test)
ridge_predict=ridge_pipeline.predict(X_test)
lasso_predict=lasso_pipeline.predict(X_test)
print(linear_predict)
print(ridge_predict)
print(lasso_predict)

[23.16849838 26.07517666 19.77753392 28.97080811 20.42715292 24.75402984
 15.4451151  27.60543697 31.93246177 15.19185124 28.43668177 -0.39250166
  2.38198431  6.91790077  8.5364988   5.93251786 21.25274491 19.48562462
 20.37202966 46.20113581]
[23.16908654 26.07501222 19.77866523 28.96997061 20.42807329 24.75405995
 15.44721508 27.60518951 31.93101491 15.1940806  28.43626329 -0.38729763
  2.38661748  6.92190979  8.53986568  5.93672962 21.25380452 19.48712509
 20.37311592 46.19698499]
[23.15954766 26.10579483 19.82177598 29.01809057 20.49291814 24.81043459
 15.47580112 27.56395918 31.9733372  15.20015665 28.39340573 -0.28169663
  2.4867872   6.92117697  8.62798744  5.93792577 21.2250101  19.43878417
 20.39205713 46.14235787]


In [9]:
print("y:", y.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("linear_predict:", linear_predict.shape)

y: (100,)
y_train: (80,)
y_test: (20,)
X_train: (80, 2)
X_test: (20, 2)
linear_predict: (20,)


In [10]:
from sklearn.metrics import mean_absolute_error,mean_squared_error, r2_score

print("Linear Regression")
print(mean_absolute_error(y_test, linear_predict))
print(mean_squared_error(y_test,linear_predict))
print(r2_score(y_test,linear_predict))

print("----")
print("Ridge ")
print(mean_absolute_error(y_test, ridge_predict))
print(mean_squared_error(y_test,ridge_predict))
print(r2_score(y_test,ridge_predict))

print("----")
print("Lasso ")
print(mean_absolute_error(y_test, lasso_predict))
print(mean_squared_error(y_test,lasso_predict))
print(r2_score(y_test,lasso_predict))

Linear Regression
3.3536803784257763
13.668697416219555
0.8579668760832718
----
Ridge 
3.3535299098997187
13.66301224219399
0.8580259514290984
----
Lasso 
3.3587754655343955
13.628469444267878
0.8583848899108624


| Model  | Alpha | MAE_mean | MAE_std | MSE_mean | MSE_std | Notes         |
| ------ | ----- | -------- | ------- | -------- | ------- | ------------- |
| Linear | –     | 3.193    | 0.195   | 13.489   | 1.712   | baseline      |
| Ridge  | 1.0   | 3.193    | 0.205   | 13.477   | 1.751   | default alpha |
| Lasso  | 1.0   | 3.244    | 0.211   | 13.871   | 1.916   | default alpha |

| Model  | Alpha | MAE_mean | MAE_std | MSE_mean | MSE_std |
| ------ | ----- | -------- | ------- | -------- | ------- |
| Linear | –     | 3.041    | 0.210   | 12.521   | 1.857   |
| Ridge  | 0.1   | 3.041    | 0.209   | 12.520   | 1.854   |
| Lasso  | 0.1   | 3.045    | 0.203   | 12.519   | 1.823   |

In [11]:

mae_scores = cross_val_score(
    estimator=linear_pipeline,
    X=X_train,
    y=y_train,
    cv=5,
    scoring="neg_mean_absolute_error"
)
mae_scores = np.abs(mae_scores)
MAE_mean = mae_scores.mean()
MAE_std = mae_scores.std()

mse_scores = cross_val_score(
    estimator=linear_pipeline,
    X=X_train,
    y=y_train,
    cv=5,
    scoring="neg_mean_squared_error"
)
mse_scores = np.abs(mse_scores)

MSE_mean = mse_scores.mean()

MSE_std = mse_scores.std()

In [12]:
print("mae_scores: ",mae_scores)
print("MAE_mean: ",MAE_mean)
print("MAE_std: ",MAE_std)

print("mse_scores: ",mse_scores)
print("MSE_mean: ",MSE_mean)
print("MSE_std: ",MSE_std)

mae_scores:  [3.26011467 3.55968627 2.43304755 3.31313412 2.89180562]
MAE_mean:  3.0915576449386952
MAE_std:  0.3924763517093573
mse_scores:  [12.69892538 17.10558822  8.94767718 13.62230101 11.80672908]
MSE_mean:  12.836244172771998
MSE_std:  2.6474476382468914


In [13]:

mae_scores = cross_val_score(
    estimator=ridge_pipeline,
    X=X_train,
    y=y_train,
    cv=5,
    scoring="neg_mean_absolute_error"
)
mae_scores = np.abs(mae_scores)
MAE_mean = mae_scores.mean()
MAE_std = mae_scores.std()

mse_scores = cross_val_score(
    estimator=ridge_pipeline,
    X=X_train,
    y=y_train,
    cv=5,
    scoring="neg_mean_squared_error"
)
mse_scores = np.abs(mse_scores)

MSE_mean = mse_scores.mean()

MSE_std = mse_scores.std()

In [14]:
print("mae_scores: ",mae_scores)
print("MAE_mean: ",MAE_mean)
print("MAE_std: ",MAE_std)

print("mse_scores: ",mse_scores)
print("MSE_mean: ",MSE_mean)
print("MSE_std: ",MSE_std)

mae_scores:  [3.26088532 3.55918791 2.43187655 3.31310184 2.89188667]
MAE_mean:  3.0913876569851273
MAE_std:  0.3928051053696775
mse_scores:  [12.70745466 17.09450217  8.94155363 13.62442312 11.80923003]
MSE_mean:  12.83543272244422
MSE_std:  2.645522373737559


In [15]:

mae_scores = cross_val_score(
    estimator=lasso_pipeline,
    X=X_train,
    y=y_train,
    cv=5,
    scoring="neg_mean_absolute_error"
)
mae_scores = np.abs(mae_scores)
MAE_mean = mae_scores.mean()
MAE_std = mae_scores.std()

mse_scores = cross_val_score(
    estimator=lasso_pipeline,
    X=X_train,
    y=y_train,
    cv=5,
    scoring="neg_mean_squared_error"
)
mse_scores = np.abs(mse_scores)

MSE_mean = mse_scores.mean()

MSE_std = mse_scores.std()

In [16]:
print("mae_scores: ",mae_scores)
print("MAE_mean: ",MAE_mean)
print("MAE_std: ",MAE_std)

print("mse_scores: ",mse_scores)
print("MSE_mean: ",MSE_mean)
print("MSE_std: ",MSE_std)

mae_scores:  [3.28284337 3.55815133 2.40830867 3.29717555 2.89340219]
MAE_mean:  3.087976222383375
MAE_std:  0.4006517749705232
mse_scores:  [12.93029082 17.04987735  8.82753738 13.50605017 11.82275196]
MSE_mean:  12.827301536944287
MSE_std:  2.6573929376448437
